# Protein abundance change

This notebook runs the analyses on protein abundance changes between variant and reference alleles using Cell Painting assays

In [ ]:
### imports
import os
import polars as pl
import numpy as np
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

## Disable truncation globally
pl.Config.set_tbl_rows(20)  # Show all rows
pl.Config.set_tbl_cols(40)  # Show all columns

%matplotlib inline

## Threshold for training imbalance: 
## filtered out any variant-reference pair where there was a class imbalance of more than 3-fold
thresh = 3

# min_class_num = 2
# ## define control types
# TC = ["EGFP"]
# NC = ["RHEB", "MAPK9", "PRKACB", "SLIRP"]
# PC = ["ALK", "ALK_Arg1275Gln", "PTK2B"]
# cNC = ["Renilla"]
# cPC = ["KRAS", "PTK2B", "GHSR", "ABL1", "BRD4", "OPRM1", "RB1", "ADA", "WT PMP22", "LYN", "TNF", 
#        "CYP2A6", "CSK", "PAK1", "ALDH2", "CHRM3", "KCNQ2", "ALK T1151M", "PRKCE", "LPAR1", "PLP1"]

## XGB classifier metrics

In [2]:
# Paths
metrics_dir = "../1_snakemake_pipeline/2025_varchamp_snakemake/snakemake_pipeline/outputs/analyses/{}/profiles_tcdropped_filtered_var_mad_outlier_featselect"
batches = ["2024_01_23_Batch_7", "2024_02_06_Batch_8"]

metrics_df, metrics_wtvar = pl.DataFrame(), pl.DataFrame()
for batch in batches:
    met_dir = metrics_dir.format(batch)
    metrics_df_batch = pl.read_csv(f"{met_dir}/metrics.csv")
    metrics_wtvar_batch = pl.read_csv(f"{met_dir}/metrics_summary.csv")

    metrics_df = pl.concat([metrics_df, metrics_df_batch])
    metrics_wtvar = pl.concat([metrics_wtvar, metrics_wtvar_batch])

In [5]:
## get the localization metrics
local_wtvar = metrics_wtvar.filter(pl.col("Classifier_type") == "localization")

## get the alleles and their reference genes
local_wtvar = local_wtvar.with_columns(
    pl.when(pl.col("AUROC_mean") > pl.col("AUROC_thresh"))
    .then(1)
    .otherwise(0)
    .alias("mislocalized")
).unique()
misloc_alleles = (
    local_wtvar
    .select("allele_0")
    .to_series()
    .to_list()
)
misloc_wt = (
    local_wtvar
    .with_columns(pl.col("allele_0").str.replace("_.*", "").alias("wt"))
    .select("wt")
    .to_series()
    .unique()
    .to_list()
)
all_alleles = misloc_alleles + misloc_wt

## get individual classifiers pass the training imbalance threshold
ind_class = metrics_df.filter(
    (~pl.col("Metadata_Control"))
    & (pl.col("Training_imbalance") < thresh)
    & (
        (pl.col("Full_Classifier_ID").str.contains("true"))
        & pl.col("allele_0").is_in(misloc_alleles)
    )
)

## CellProfiler Features

### Get the CP features for cells that passed the QC

In [7]:
pass_qc_prof_dir = "../1_snakemake_pipeline/2025_varchamp_snakemake/snakemake_pipeline/outputs/batch_profiles/{}/profiles_tcdropped_filtered_var_mad_outlier_featselect.parquet"

# Get meta features
b7_alleles = (
    pl.scan_parquet(
        pass_qc_prof_dir.format(batches[0])
    )
    .filter(pl.col("Metadata_gene_allele").is_in(all_alleles))
    .with_columns(
        pl.concat_str(
            [
                "Metadata_Plate",
                "Metadata_Well",
                "Metadata_ImageNumber",
                "Metadata_ObjectNumber",
            ],
            separator="_",
        ).alias("Metadata_CellID")
    )
    .select([
        "Metadata_CellID",
        "Metadata_gene_allele",
        "Metadata_Well",
        "Metadata_Plate",
    ])
)

b8_alleles = (
    pl.scan_parquet(
        pass_qc_prof_dir.format(batches[1])
    )
    .filter(pl.col("Metadata_gene_allele").is_in(all_alleles))
    .with_columns(
        pl.concat_str(
            [
                "Metadata_Plate",
                "Metadata_Well",
                "Metadata_ImageNumber",
                "Metadata_ObjectNumber",
            ],
            separator="_",
        ).alias("Metadata_CellID")
    )
    .select([
        "Metadata_CellID",
        "Metadata_gene_allele",
        "Metadata_Well",
        "Metadata_Plate",
    ])
)

cell_alleles = pl.concat([b7_alleles.collect(), b8_alleles.collect()])

### Get the Cells_Intensity CP features per all cells

Merge the Cells_Intensity features to the pass-QC cells

In [8]:
prof_dir = "../1_snakemake_pipeline/2025_varchamp_snakemake/snakemake_pipeline/outputs/batch_profiles/{}/profiles.parquet"

b7_prof = pl.scan_parquet(
    prof_dir.format(batches[0])
).with_columns(
    pl.concat_str(
        [
            "Metadata_Plate",
            "Metadata_Well",
            "Metadata_ImageNumber",
            "Metadata_ObjectNumber",
        ],
        separator="_",
    ).alias("Metadata_CellID")
)
b8_prof = pl.scan_parquet(
    prof_dir.format(batches[1])
).with_columns(
    pl.concat_str(
        [
            "Metadata_Plate",
            "Metadata_Well",
            "Metadata_ImageNumber",
            "Metadata_ObjectNumber",
        ],
        separator="_",
    ).alias("Metadata_CellID")
)

gfp_int = [i for i in b7_prof.collect_schema().names() if "Cells_Intensity" in i]
gfp_int = ["Metadata_CellID"] + [i for i in gfp_int if "GFP" in i]

profiles = pl.concat([
    b7_prof.select(gfp_int).collect(),
    b8_prof.select(gfp_int).collect(),
])

# This method of joining ensures that only cells past filters are included in the abundance analysis
profiles = cell_alleles.join(profiles, on="Metadata_CellID", how="left")

### Aggregate the cells to well profiles

In [21]:
well_profiles = (
    profiles.group_by(["Metadata_Plate", "Metadata_Well", "Metadata_gene_allele"])
    .agg(
        pl.col(col).median().alias(col)
        for col in profiles.columns
        if not col.startswith("Metadata_")
    )
    # .drop("Metadata_CellID")
    .unique()
)

well_profiles

Metadata_Plate,Metadata_Well,Metadata_gene_allele,Cells_Intensity_IntegratedIntensityEdge_GFP,Cells_Intensity_IntegratedIntensity_GFP,Cells_Intensity_LowerQuartileIntensity_GFP,Cells_Intensity_MADIntensity_GFP,Cells_Intensity_MassDisplacement_GFP,Cells_Intensity_MaxIntensityEdge_GFP,Cells_Intensity_MaxIntensity_GFP,Cells_Intensity_MeanIntensityEdge_GFP,Cells_Intensity_MeanIntensity_GFP,Cells_Intensity_MedianIntensity_GFP,Cells_Intensity_MinIntensityEdge_GFP,Cells_Intensity_MinIntensity_GFP,Cells_Intensity_StdIntensityEdge_GFP,Cells_Intensity_StdIntensity_GFP,Cells_Intensity_UpperQuartileIntensity_GFP
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2024_02_02_B8A1R2_P3T1""","""F13""","""PLA2G6_Glu567Lys""",1.088361,29.757551,0.003006,0.000958,1.778141,0.003765,0.010545,0.002209,0.00419,0.003881,0.001575,0.001575,0.000386,0.001539,0.005221
"""2024_01_18_B7A1R1_P3T1""","""E18""","""PDGFRB_Leu658Pro""",0.816713,17.543378,0.002124,0.00016,0.737983,0.002526,0.005146,0.001976,0.002366,0.002246,0.00149,0.001439,0.000194,0.000312,0.002432
"""2024_01_17_B7A1R1_P2T4""","""N23""","""KRT2_Gly219Asp""",0.822382,26.520131,0.002154,0.000726,3.283171,0.005336,0.044338,0.002002,0.004215,0.002917,0.001408,0.00139,0.000529,0.00401,0.004012
"""2024_02_01_B8A1R2_P2T4""","""P14""","""LIG4""",0.751569,13.316699,0.001747,0.00008,0.344393,0.00216,0.002357,0.001794,0.001826,0.001824,0.00146,0.001408,0.000115,0.000121,0.001904
"""2024_01_18_B7A1R1_P3T1""","""O21""","""MVK_His20Gln""",3.066807,126.413021,0.009516,0.005284,4.29441,0.03273,0.049263,0.007071,0.016682,0.014629,0.001776,0.001754,0.006386,0.00917,0.021414
"""2024_02_02_B8A1R2_P4T1""","""I22""","""THAP1_Val193Ile""",0.709246,16.79242,0.001855,0.000217,1.926055,0.002471,0.005855,0.001789,0.002298,0.002038,0.001412,0.00138,0.000156,0.000744,0.002321
"""2024_01_19_B7A1R1_P4T1""","""D11""","""TTPA_Arg192His""",2.377201,97.067289,0.006756,0.004383,4.471228,0.021639,0.042846,0.00568,0.012854,0.011705,0.00191,0.001866,0.004047,0.007299,0.017859
"""2024_01_17_B7A1R1_P1T2""","""E18""","""CDC73""",0.670741,18.703289,0.001737,0.000121,3.738663,0.002187,0.010859,0.00176,0.002734,0.001845,0.001428,0.001372,0.000123,0.00206,0.00203
"""2024_01_17_B7A1R1_P1T1""","""H04""","""DES_Ala237Thr""",0.829459,41.964079,0.001987,0.000816,5.152037,0.00548,0.165331,0.001986,0.006716,0.00269,0.001391,0.001366,0.000614,0.015711,0.00495


## Calculate abundance hits

Use paired t-test to call abundance hits

In [23]:
from scipy.stats import shapiro
import re
from scipy.stats import wilcoxon
from scipy.stats import ttest_rel
import pandas as pd


# Convert letter rows to numbers
def well_to_coordinates(well):
    row_letter, col_number = re.match(r"([A-P])(\d{2})", well).groups()
    row_index = ord(row_letter) - ord('A') + 1  # Convert 'A'->1, 'B'->2, ..., 'P'->16
    col_index = int(col_number)  # Convert string column to integer
    return well, row_index, col_index


# Compute distances from edges and find the most centered well
def compute_distance(row, col):
    return min(row - 1, 16 - row, col - 1, 24 - col)  # Distance from nearest edge
    

def paired_ttest(dat, reference: str, var: str):
    ## perform stat test
    dat = dat.dropna().sort_values(["Metadata_Plate","Metadata_gene_allele"])

    # Assuming well_abun_stats is a DataFrame with columns 'reference_abundance' and 'variant_abundance'
    reference_abundance = dat[dat["Metadata_gene_allele"]==reference]["Cells_Intensity_MedianIntensity_GFP"].values
    variant_abundance = dat[dat["Metadata_gene_allele"]==var]["Cells_Intensity_MedianIntensity_GFP"].values
    t_stat, p_val = ttest_rel(variant_abundance, reference_abundance)

    # Calculate Cohen's d
    mean_diff = np.mean(variant_abundance) - np.mean(reference_abundance)
    pooled_std = np.sqrt((np.std(variant_abundance, ddof=1) ** 2 + np.std(reference_abundance, ddof=1) ** 2) / 2)
    cohen_d = mean_diff / pooled_std
    summary_df = pl.DataFrame(
        {
            "t_stat": t_stat,
            "p_val": p_val,
            "cohen_d": cohen_d
        }
    )
    summary_df = summary_df.with_columns(
        pl.lit(reference).alias("Gene"), pl.lit(var).alias("Variant")
    )
    return summary_df



well_abun_stats = []
for allele in tqdm(misloc_alleles):
    temp_class = ind_class.filter(pl.col("allele_0") == allele)
    temp_prof = well_profiles.filter(
        (pl.col("Metadata_Plate").is_in(temp_class.select("Plate").to_series()))
        & (
            pl.col("Metadata_Well").is_in(temp_class.select("well_1"))
            | pl.col("Metadata_Well").is_in(temp_class.select("well_0"))
        )
    )
    reference = allele.split("_")[0]

    temp_prof = temp_prof.to_pandas().dropna().sort_values(["Metadata_Plate","Metadata_gene_allele"])
    var_profiles = temp_prof[temp_prof["Metadata_gene_allele"]==allele]
    ref_profiles = temp_prof[temp_prof["Metadata_gene_allele"]==reference]
    
    # print(temp_prof)
    # diff = temp_prof[temp_prof["Metadata_gene_allele"]==allele]["Cells_Intensity_IntegratedIntensityEdge_GFP"].values - \
    #        temp_prof[temp_prof["Metadata_gene_allele"]==reference]["Cells_Intensity_IntegratedIntensityEdge_GFP"].values

    # try:
    if (ref_profiles.shape[0] > var_profiles.shape[0]):
        ref_al_wells = ref_profiles["Metadata_Well"].unique()
        well_coords = [well_to_coordinates(w) for w in set([ref_well_pl for ref_well_pl in ref_al_wells])]
        # Sort wells by max distance from edges (descending)
        most_centered_well = max(well_coords, key=lambda x: compute_distance(x[1], x[2]))[0]
        ref_profiles = temp_prof[(temp_prof["Metadata_gene_allele"]==reference)&(temp_prof["Metadata_Well"]==most_centered_well)]
        
    if (ref_profiles.shape[0] < var_profiles.shape[0]):
        ref_al_wells = var_profiles["Metadata_Well"].unique()
        well_coords = [well_to_coordinates(w) for w in set([ref_well_pl for ref_well_pl in ref_al_wells])]
        # Sort wells by max distance from edges (descending)
        most_centered_well = max(well_coords, key=lambda x: compute_distance(x[1], x[2]))[0]
        var_profiles = temp_prof[(temp_prof["Metadata_gene_allele"]==allele)&(temp_prof["Metadata_Well"]==most_centered_well)]

    well_abun_stats.append(
        paired_ttest(
            dat=pd.concat([ref_profiles, var_profiles]),
            reference=reference,
            var=allele
        )
    )

well_abun_stats = pl.concat(well_abun_stats, how="vertical")
well_abun_stats = well_abun_stats.rename({"t_stat": "U2OS_t"})
well_abun_stats = well_abun_stats.sort(["Gene", "Variant", "U2OS_t", "p_val", "cohen_d"])
well_abun_stats

100%|██████████| 1126/1126 [00:44<00:00, 25.33it/s]


U2OS_t,p_val,cohen_d,Gene,Variant
f64,f64,f64,str,str
1.045202,0.372731,0.421038,"""ACACB""","""ACACB_Val958Met"""
-7.045782,0.005876,-2.871654,"""ACSF3""","""ACSF3_Ala197Thr"""
-3.177852,0.050177,-1.535075,"""ACSF3""","""ACSF3_Arg10Trp"""
-4.021488,0.00505,-1.550507,"""ACSF3""","""ACSF3_Arg471Trp"""
-4.021488,0.00505,-1.550507,"""ACSF3""","""ACSF3_Arg471Trp"""
-3.393741,0.01154,-1.503345,"""ACSF3""","""ACSF3_Arg558Trp"""
-3.393741,0.01154,-1.503345,"""ACSF3""","""ACSF3_Arg558Trp"""
-3.888603,0.005986,-1.497214,"""ACSF3""","""ACSF3_Asp457Asn"""
-3.888603,0.005986,-1.497214,"""ACSF3""","""ACSF3_Asp457Asn"""


In [24]:
well_abun_stats.write_csv("../../3_outputs/abundance/abundance_ttest_stats.csv")